# Swin student results

Compare the teacher and all student evaluations produced by `run_swin_experiments.sh`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULT_DIR = Path("results/swin_experiments/evaluations")

In [ ]:
records = {}
for path in sorted(RESULT_DIR.glob("*.json")):
    result = json.loads(path.read_text())
    records[path.stem] = {
        "strict_macro_field_f1": result["score"],
        "total_parameters": result["parameters"]["total"],
        "encoder_parameters": result["parameters"]["encoder"],
        "decoder_parameters": result["parameters"]["decoder"],
    }

comparison = pd.DataFrame.from_dict(records, orient="index")
comparison

In [ ]:
teacher = comparison.loc["teacher"]
comparison["parameter_reduction"] = (
    1 - comparison["total_parameters"] / teacher["total_parameters"]
)
comparison["score_change"] = (
    comparison["strict_macro_field_f1"] - teacher["strict_macro_field_f1"]
)
comparison.sort_values("total_parameters", ascending=False)

In [ ]:
ordered = comparison.sort_values("total_parameters")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ordered["strict_macro_field_f1"].plot.barh(ax=axes[0])
axes[0].set(title="Extraction quality", xlabel="Strict macro field F1", xlim=(0, 1))

(ordered["total_parameters"] / 1e6).plot.barh(ax=axes[1])
axes[1].set(title="Model size", xlabel="Millions of parameters")

plt.tight_layout()

In [ ]:
students = comparison.drop(index="teacher")
students.plot.scatter(
    x="total_parameters",
    y="strict_macro_field_f1",
    figsize=(7, 5),
)
plt.axhline(
    teacher["strict_macro_field_f1"], color="black", linestyle="--", label="teacher F1"
)
plt.xlabel("Parameters")
plt.ylabel("Strict macro field F1")
plt.legend()
plt.tight_layout()